# SAMap supplementary figure, panel g: observed statistics against 1,000 shuffles of each design

Each strip shows one statistic over 1,000 transfers of shuffled mouse pseudotime (light grey: shuffled across all
mouse nuclei; dark grey: shuffled within each mouse compartment), its median (short black line) and the value for
the transfer of record (long black line, value printed). Top row: Spearman's rho between transferred and human
pseudotime, over all nuclei and within each compartment. Bottom row: Moran's I of the transferred values in the
human sections, over all nuclei and within each compartment, with spatial neighbors found within each timepoint's
section (its three 10x lanes together), averaged over the two timepoints. All statistics use all nuclei.

**Inputs**: `null_replicates.csv` (rule `samap_ptnull`: one row per transfer, the observed one first) and, for the
check that the permutation run used the transfer of record, `example_transfers.csv` and `human_cells.csv`.
**Outputs**, in `OUT`: `sfig_samap_panel_g.png` (300 dpi), `sfig_samap_panel_g_600dpi.png`,
`sfig_samap_panel_g.svg` (text kept as text) and `sfig_samap_panel_g.provenance.json`.
`sfig_samap_compose.ipynb` places the panel in the figure and adds its letter.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_g_permutation_statistics.ipynb \
    results/notebooks/samap/sfig_samap_panel_g.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
NULL_REPLICATES = "results/samap_figures/ptnull/null_replicates.csv"
EXAMPLE_TRANSFERS = "results/samap_figures/ptnull/example_transfers.csv"
HUMAN_CELLS = "results/samap_figures/tables/human_cells.csv"
OUT = "results/figures/samap_supp/panels"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_g_permutation_statistics.ipynb"  # this notebook, for the provenance record

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from IPython.display import Image, display

import sfig_samap_style as style

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_g")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
LETTER = "g"
TITLE = "The transfer exceeds every shuffle\nin ordering and spatial structure"
WIDTH_SHOWN = 380  # px, the PNG shown in this notebook

# Rows: label, y limits and ticks. Columns: header. Each (row, column) maps to a null_replicates.csv column.
STAT_ROWS = {
    "rho": ("Spearman ρ with\nhuman pseudotime", (-0.3, 1.0), [0, 0.5, 1.0]),
    "moran": ("Moran's I (spatial\nautocorrelation)", (-0.05, 0.8), [0, 0.4, 0.8]),
}
STAT_COLUMNS = {"all": "all cells", "decidual": "decidual", "fibroblast": "fibroblasts"}
STAT_KEYS = {
    ("rho", "all"): "rho_all",
    ("rho", "decidual"): "rho_decidual",
    ("rho", "fibroblast"): "rho_fibroblast",
    ("moran", "all"): "moran_all",
    ("moran", "decidual"): "moran_within_decidual",
    ("moran", "fibroblast"): "moran_within_fibroblast",
}

# The two shuffle designs differ by lightness only and never share a strip; the legend keys them.
NULL_COLORS = {"uniform": "#b8b8b8", "within_compartment": "#6b6b6b"}
NULL_LABELS = {
    "uniform": "1,000 shuffles across all mouse cells",
    "within_compartment": "1,000 shuffles within mouse compartment",
}
JITTER = 0.22  # strip half-width, in x units
JITTER_SEED = 0  # horizontal jitter only; no statistic depends on it

## Statistics

In [ ]:
def compare_to_shuffles(replicates: pd.DataFrame) -> pd.DataFrame:
    """Observed value, median of the shuffles and shuffles at or above the observed value, per design and statistic."""
    observed = replicates[replicates.design == "observed"].iloc[0]
    rows = []
    for design in NULL_COLORS:
        shuffles = replicates[replicates.design == design]
        for stat in STAT_KEYS.values():
            rows.append({
                "design": design,
                "statistic": stat,
                "observed": observed[stat],
                "median_of_shuffles": shuffles[stat].median(),
                "shuffles_at_or_above_observed": int((shuffles[stat] >= observed[stat]).sum()),
                "shuffles": len(shuffles),
            })
    return pd.DataFrame(rows)

## Figure

In [ ]:
def draw_null_strips(replicates: pd.DataFrame):
    observed = replicates[replicates.design == "observed"].iloc[0]
    fig, width_mm, height_mm = style.new_panel(LETTER)
    left, right, col_gap, axes_h_mm, row_gap = 17.0, 1.0, 2.0, 22.0, 4.5
    axes_w_mm = (width_mm - left - right - 2 * col_gap) / 3
    rng = np.random.default_rng(JITTER_SEED)

    for i, (row_key, (row_label, ylim, yticks)) in enumerate(STAT_ROWS.items()):
        top_mm = height_mm - 13.5 - i * (axes_h_mm + row_gap)
        for j, (col_key, col_label) in enumerate(STAT_COLUMNS.items()):
            ax = fig.add_axes([(left + j * (axes_w_mm + col_gap)) / width_mm, (top_mm - axes_h_mm) / height_mm,
                               axes_w_mm / width_mm, axes_h_mm / height_mm])
            stat = STAT_KEYS[(row_key, col_key)]

            # One jittered strip per design, at x = 0 and 1, with its median.
            for position, design in enumerate(NULL_COLORS):
                values = replicates.loc[replicates.design == design, stat].to_numpy()
                x = position + rng.uniform(-JITTER, JITTER, values.size)
                ax.scatter(x, values, s=0.8, color=NULL_COLORS[design], linewidths=0, rasterized=True)
                ax.plot([position - 0.3, position + 0.3], [np.median(values)] * 2, color="black", lw=0.6)

            # The transfer of record across both strips, its value printed at the left end.
            ax.axhline(observed[stat], color="black", lw=1.1)
            ax.text(-0.45, observed[stat] + 0.015 * (ylim[1] - ylim[0]), f"{observed[stat]:.2f}",
                    ha="left", va="bottom", fontsize=7)

            # Shared y range per row, so the columns compare directly; design identity is carried by shade.
            ax.set_xlim(-0.5, 1.5)
            ax.set_ylim(*ylim)
            ax.set_xticks([])
            ax.set_yticks(yticks)
            if i == 0:
                ax.set_title(col_label, fontsize=8, pad=2)
            if j == 0:
                ax.set_ylabel(row_label, fontsize=8)
            else:
                ax.set_yticklabels([])

    fig.legend(
        handles=[
            Line2D([], [], color="black", lw=1.1, label="observed transfer"),
            *[Line2D([], [], ls="none", marker="o", ms=3, color=color, label=NULL_LABELS[design])
              for design, color in NULL_COLORS.items()],
            Line2D([], [], color="black", lw=0.6, label="median of shuffles"),
        ],
        loc="lower left",
        bbox_to_anchor=(3.0 / width_mm, 0.6 / height_mm),
        frameon=False,
        fontsize=8,
        handlelength=1.4,
        labelspacing=0.3,
        borderaxespad=0,
    )
    style.panel_title(fig, width_mm, height_mm, TITLE)
    return fig

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

paths = style.panel_paths(repo / OUT, LETTER)
style.refuse_overwrite(paths.values(), FORCE)
paths["png"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
replicates = pd.read_csv(repo / NULL_REPLICATES)
transfers = pd.read_csv(repo / EXAMPLE_TRANSFERS).set_index("cell")
human = pd.read_csv(repo / HUMAN_CELLS, index_col="cell")
style.check_same_transfer(transfers, human)

counts = replicates.design.value_counts().to_dict()
LOG.info("Transfers per design: %s", counts)
comparison = compare_to_shuffles(replicates)
display(comparison.round(4))

In [ ]:
font = style.choose_font()
style.apply_style(font)
LOG.info("Font: %s", font)

fig = draw_null_strips(replicates)

style.save_panel(fig, LETTER, paths)
plt.close(fig)
LOG.info("Wrote %s", [str(path) for name, path in paths.items() if name != "provenance"])

# The saved PNG rather than the live figure, so what shows here is the file itself.
display(Image(filename=str(paths["png"]), width=WIDTH_SHOWN))

In [ ]:
observed = replicates[replicates.design == "observed"].iloc[0]
record = style.provenance(
    notebook,
    parameters={
        "REPO": str(repo),
        "NULL_REPLICATES": NULL_REPLICATES,
        "EXAMPLE_TRANSFERS": EXAMPLE_TRANSFERS,
        "HUMAN_CELLS": HUMAN_CELLS,
        "OUT": OUT,
        "FORCE": FORCE,
    },
    inputs={
        "null_replicates": repo / NULL_REPLICATES,
        "example_transfers": repo / EXAMPLE_TRANSFERS,
        "human_cells": repo / HUMAN_CELLS,
    },
    outputs=[paths["png"], paths["png_600dpi"], paths["svg"]],
    font=font,
)
record["panel"] = style.panel_spec(LETTER)
record["permutation_observed"] = {stat: round(float(observed[stat]), 4) for stat in STAT_KEYS.values()}
record["permutation_replicates"] = {design: int(counts.get(design, 0)) for design in NULL_COLORS}
record["comparison"] = comparison.round(4).to_dict("records")
style.write_json(paths["provenance"], record)
LOG.info("Wrote %s", paths["provenance"])